# Silver - Espelho
Coisas permitidas:
- tipagem: string vira timestamp, int, date, etc
- legibilidade: quebrar timestamp em data e hora
- metadados: comment em toda coluna, tags na tabela
- unificação: dois cadastros do mesmo assunto no mesmo lugar, com a origem por registro
- aritmética pura: atraso = real - previsto

Coisas que não são permitidas:
- filtro / where de negócio
- group by / agregação
- limiar, flag, classificação

Silver precisa servir **toda** análise. Regras de negócio pertencem na Gold.

In [0]:
display(spark.sql("""
    SELECT 
        COUNT(*) AS linhas,
        SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS partida_real_null_de_verdade,
        SUM(CASE WHEN partida_real = 'null' THEN 1 ELSE 0 END) AS partida_real_string_null,
        SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END) AS partida_prevista_string_null,
        SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END) AS com_fracao_de_segundo
    FROM voe_bem.bronze.vra
"""))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS voe_bem.silver

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE voe_bem.silver.vra AS
    WITH tipado AS (
        SELECT
            icao_empresa_aerea,
            numero_voo,
            codigo_autorizacao,
            codigo_tipo_linha,
            icao_aerodromo_origem,
            icao_aerodromo_destino,
            try_cast(nullif(partida_prevista, 'null') AS TIMESTAMP) AS partida_prevista,
            try_cast(nullif(partida_real, 'null') AS TIMESTAMP) AS partida_real,
            try_cast(nullif(chegada_prevista, 'null') AS TIMESTAMP) AS chegada_prevista,
            try_cast(nullif(chegada_real, 'null') AS TIMESTAMP) AS chegada_real,
            situacao_voo,
            nullif(codigo_justificativa, 'N/A') AS codigo_justificativa,
            _arquivo_origem,
            _ingerido_em
        FROM voe_bem.bronze.vra
    )
    SELECT
        icao_empresa_aerea,
        numero_voo,
        codigo_autorizacao,
        codigo_tipo_linha,
        icao_aerodromo_origem,
        icao_aerodromo_destino,
        
        partida_prevista,
        CAST(partida_prevista AS DATE)                              AS partida_prevista_data,
        date_format(partida_prevista, 'HH:mm')                     AS partida_prevista_hora,
        
        partida_real,
        CAST(partida_real AS DATE)                                  AS partida_real_data,
        date_format(partida_real, 'HH:mm')                         AS partida_real_hora,
        
        chegada_prevista,
        CAST(chegada_prevista AS DATE)                              AS chegada_prevista_data,
        date_format(chegada_prevista, 'HH:mm')                      AS chegada_prevista_hora,

        chegada_real,
        CAST(chegada_real AS DATE)                                  AS chegada_real_data,
        date_format(chegada_real, 'HH:mm')                         AS chegada_real_hora,
        
        situacao_voo,
        codigo_justificativa,
        
        -- aritmetica pura: subtracao de colunas da propria linha, sem decisao
        CAST(timestampdiff(MINUTE, partida_prevista, partida_real) AS INT) AS atraso_partida_min,
        CAST(timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS atraso_chegada_min,
        CAST(timestampdiff(MINUTE, partida_prevista, partida_real) - timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutos_recuperados,

        _arquivo_origem,
        _ingerido_em,
        current_timestamp() AS _transformado_em
    FROM tipado
""")

In [0]:
display(spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM voe_bem.bronze.vra) AS bronze_vra,
        (SELECT COUNT(*) FROM voe_bem.silver.vra) AS silver_vra,
        (SELECT COUNT(*) FROM voe_bem.bronze.vra)
        - (SELECT COUNT(*) FROM voe_bem.silver.vra) AS diferenca
"""))

In [0]:
display(spark.sql("""
    SELECT
        COUNT(partida_prevista)        AS partida_prevista_ok,
        COUNT(partida_real)            AS partida_real_ok,
        COUNT(chegada_prevista)        AS chegada_prevista_ok,
        COUNT(chegada_real)            AS chegada_real_ok,
        COUNT(atraso_partida_min)      AS atraso_partida_ok,
        COUNT(minutos_recuperados)     AS minutos_recuperados_ok
    FROM voe_bem.silver.vra
"""))

In [0]:
display(spark.sql("""
    SELECT icao_empresa_aerea, numero_voo, icao_aerodromo_origem, icao_aerodromo_destino,
        partida_prevista, partida_prevista_data, partida_prevista_hora,
        atraso_partida_min, atraso_chegada_min, minutos_recuperados, situacao_voo
    FROM voe_bem.silver.vra
    ORDER BY partida_prevista
    LIMIT 5
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE voe_bem.silver.empresas AS
SELECT
    icao,
    estrangeira AS sigla_iata,
    razao as razao_social,
    servico,
    cidade,
    uf,
    ativa as situacao,
    'nacional' AS origem_cadastro,
    'voe_bem.bronze.empresas_nacionais' as _arquivo_origem,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voe_bem.bronze.empresas_nacionais
UNION ALL
SELECT
    icao,
    estrangeira AS sigla_iata,
    razao as razao_social,
    servico,
    cidade,
    uf,
    ativa as situacao,
    'estrangeira' AS origem_cadastro,
    'voe_bem.bronze.empresas_nacionais' as _arquivo_origem,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voe_bem.bronze.empresas_estrangeiras
""")

display(spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM voe_bem.bronze.empresas_nacionais) AS bronze_nacionais,
        (SELECT COUNT(*) FROM voe_bem.bronze.empresas_estrangeiras) AS bronze_estrangeiras,
        (SELECT COUNT(*) FROM voe_bem.bronze.empresas_nacionais)
        + (SELECT COUNT(*) FROM voe_bem.bronze.empresas_estrangeiras) AS soma_esperada,
        (SELECT COUNT(*) FROM voe_bem.silver.empresas) AS silver_empresas
"""))


In [0]:
display(spark.sql("""
    SELECT origem_cadastro,
        COUNT(*) AS linhas,
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voe_bem.silver.empresas
    GROUP BY origem_cadastro
    ORDER BY origem_cadastro
"""))

# Aeródromos

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE voe_bem.silver.aerodromos AS
SELECT
    codigo_oaci as icao,
    ciad,
    nome,
    municipio,
    uf AS uf_nome,
    municipio_servido,
    uf_servido AS uf_servido_nome,
    latitude AS latitude_dms,
    longitude AS longitude_dms,
    try_cast(replace(altitude, ',', '.') AS DOUBLE) AS altitude_m,
    situacao,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voe_bem.bronze.aerodromos
""")

spark.sql("""
CREATE OR REPLACE TABLE voe_bem.silver.codigos_operacao AS
SELECT
    dominio,
    codigo,
    descricao,
    current_timestamp() AS _transformado_em
FROM voe_bem.bronze.codigos_operacao
""")

display(spark.sql("""
    SELECT 'aerodromos' AS tabela,
        (SELECT COUNT(*) FROM voe_bem.bronze.aerodromos) AS bronze,
        (SELECT COUNT(*) FROM voe_bem.silver.aerodromos) AS silver
    UNION ALL
    SELECT 'codigos_operacao',
        (SELECT COUNT(*) FROM voe_bem.bronze.codigos_operacao),
        (SELECT COUNT(*) FROM voe_bem.silver.codigos_operacao)
"""))


# Metadados Gerenciados

In [0]:
COMENTARIOS_VRA = {
    "icao_empresa_aerea":           "Codigo ICAO de tres letras da empresa aerea que operou a etapa. Chave para silver.empresas.",
    "numero_voo":                   "Numero de voo divulgado pela companhia. Identificador comercial, nao numerico: pode ter zero a esquerda e se repete entre datas.",
    "codigo_autorizacao":           "Codigo de autorizacao (DI) da etapa: distingue etapa regular, extra, de retorno, charter. Descricao em silver.codigos_operacao (dominio codigo_di).",
    "codigo_tipo_linha":            "Codigo de tipo de linha: N e C domesticas, I e G internacionais. Descricao em silver.codigos_operacao (dominio codigo_tipo_linha).",
    "icao_aerodromo_origem":        "Codigo ICAO do aerodromo de onde a etapa partiu. Chave para silver.aerodromos - aeroportos, estrangeiros nao constam no cadastro da ANAC.",
    "icao_aerodromo_destino":       "Codigo ICAO do aerodromo onde a etapa pousou. Mesma observacao de cobertura da origem.",
    "partida_prevista":             "Horario de partida programado pela companhia, na hora local do aeroporto de origem.",
    "partida_prevista_data":        "Data da partida programada, separada para facilitar analise por dia.",
    "partida_prevista_hora":        "Hora e minuto da partida programada (HH:mm), para facilitar analise por faixa horaria.",
    "partida_real":                 "Horario em que a aeronave efetivamente saiu. Nulo em voo cancelado, que nao chegou a partir.",
    "partida_real_data":            "Data da partida efetiva.",
    "partida_real_hora":            "Hora e minuto da partida efetiva (HH:mm).",
    "chegada_prevista":             "Horario de chegada programado, na hora local do aeroporto de destino.",
    "chegada_prevista_data":        "Data da chegada programada.",
    "chegada_prevista_hora":        "Hora e minuto da chegada programada (HH:mm).",
    "chegada_real":                 "Horario em que a aeronave efetivamente pousou. Nulo em voo cancelado.",
    "chegada_real_data":            "Data da chegada efetiva.",
    "chegada_real_hora":            "Hora e minuto da chegada efetiva (HH:mm).",
    "situacao_voo":                 "Situacao informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando nao.",
    "codigo_justificativa":         "Motivo declarado do atraso. Deixou de ser exigido pela ANAC em abril de 2020 com a revogacao da IAC 1504: vee vazio em toda a janela deste projeto.",
    "atraso_partida_min":           "Minutos entre a partida programada e a partida efetiva. Positivo e atraso, negativo e antecipacao. Aritmetica pura: nao aplica limiar de pontualidade.",
    "atraso_chegada_min":           "Minutos entre a chegada programada e a chegada efetiva. Positivo e atraso, negativo e antecipacao.",
    "minutos_recuperados":          "Minutos que a etapa recuperou em voo: atraso de partida menos atraso de chegada. Positivo significa que chegou menos atrasada do que saiu.",
    "_arquivo_origem":              "Auditoria: nome do arquivo CSV mensal da ANAC de onde a linha veio.",
    "_ingerido_em":                 "Auditoria: momento em que a linha entrou no bronze.",
    "_transformado_em":             "Auditoria: momento em que a silver foi reconstruida a partir do bronze."
}

for coluna, comentario in COMENTARIOS_VRA.items():
    spark.sql(f"ALTER TABLE voe_bem.silver.vra ALTER COLUMN {coluna} COMMENT '{comentario}'")

print(f"{len(COMENTARIOS_VRA)} colunas documentadas")

In [0]:
COMENTARIOS_EMPRESAS = {
    "icao": "Codigo ICAO de tres letras da empresa. Vazio para operadores sem codigo (aviacao geral, taxi aereo, aeroclub).",
    "sigla_iata": "Sigla de duas letras da empresa no padrao IATA, como publicada pela ANAC.",
    "razao_social": "Razao social da empresa aerea. E o nome que aparece para quem consome o produto final.",
    "servico": "Tipo de servico autorizado pela ANAC: transporte regular, nao regular, aeragricola, taxi aereo.",
    "cidade": "Municipio da sede ou do representante legal no Brasil.",
    "uf": "Sigla da unidade federativa da sede.",
    "situacao": "Situacao do registro na ANAC: ATIVA ou nao. Registro inativo permanece na tabela porque a empresa pode ter voado no periodo analisado.",
    "origem_cadastro": "De qual dos dois cadastros da ANAC este registro veio: nacional ou estrangeira. E a coluna que preserva a fronteira entre as duas fontes depois da uniao.",
    "_arquivo_origem": "Auditoria: arquivo CSV de origem.",
    "_ingerido_em": "Auditoria: momento da ingestao no bronze.",
    "_transformado_em": "Auditoria: momento da construcao da silver."
}

COMENTARIOS_AERODROMOS = {
    "icao": "Codigo ICAO (OACI) do aerodromo. Chave de ligacao com origem a destino do VRA.",
    "ciad": "Codigo de identificacao do aerodromo no cadastro da ANAC.",
    "nome": "Nome do aerodromo como publicado pela ANAC.",
    "municipio": "Municipio onde o aerodromo esta fisicamente localizado.",
    "uf_nome": "Nome de unidade federativa POR EXTENSO (Acre, Sao Paulo), nao a sigla: e assim que a ANAC publica.",
    "municipio_servido": "Municipio principal atendido pelo aerodromo, que pode ser diferente do municipio onde ele fica.",
    "uf_servido_nome": "Nome por extenso da UF do municipio servido.",
    "latitude_dms": "Latitude em graus, minutos e segundos, como publicada pela ANAC.",
    "longitude_dms": "Longitude em graus, minutos e segundos, como publicada pela ANAC.",
    "altitude_m": "Altitude do aerodromo em metros. Na origem vem com virgula decimal.",
    "situacao": "Situacao do aerodromo no cadastro da ANAC.",
    "_ingerido_em": "Auditoria: momento da ingestao no bronze.",
    "_transformado_em": "Auditoria: momento da construcao da silver."
}

COMENTARIOS_CODIGOS = {
    "dominio": "A qual coluna do VRA este codigo pertence: codigo_di ou codigo_tipo_linha.",
    "codigo": "O codigo como aparece no VRA.",
    "descricao": "Descricao oficial do codigo, curada da pagina de descricao de variaveis da ANAC.",
    "_transformado_em": "Auditoria: momento da construcao da silver."
}

for tabela, mapa in [
    ("voe_bem.silver.empresas", COMENTARIOS_EMPRESAS),
    ("voe_bem.silver.aerodromos", COMENTARIOS_AERODROMOS),
    ("voe_bem.silver.codigos_operacao", COMENTARIOS_CODIGOS),
]:
    for coluna, comentario in mapa.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    print(f"{len(mapa)} colunas comentadas em {tabela}")


In [0]:
TABELAS = {
    "voe_bem.silver.vra": (
        "Silver - espelho governado de bronze.vra. Mesmo grao (uma linha por etapa de voo) e "
        "MESMA contagem de linhas do bronze: sem filtro, sem agregacao e sem regra de negocio. "
        "Traz tipagem, data e hora separadas e as tres metricas de aritmetica pura de atraso. "
        "Pontualidade, escopo e exclusoes ficam na gold.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-VRA", "grao": "etapa_de_voo"},
    ),
    "voe_bem.silver.empresas": (
        "Silver - cadastro unificado de empresas aereas: uniao dos dois cadastros do bronze "
        "(nacionais e estrangeiras) com a coluna origem_cadastro preservando a fonte de cada registro. "
        "Contagem igual a soma exata das duas tabelas de origem.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-Operador-Aereo", "grao": "empresa"},
    ),
    "voe_bem.silver.aerodromos": (
        "Silver - espelho governado do cadastro de aerodromos publicos da ANAC. Cobre apenas "
        "aerodromos brasileiros: aeroportos estrangeiros do VRA nao constam aqui, e isso e "
        "propriedade da fonte, nao defeito.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-Aerodromos", "grao": "aerodromo"},
    ),
    "voe_bem.silver.codigos_operacao": (
        "Silver - espelho da seed table de codigos de operacao (DI e tipo de linha) com as "
        "descricoes oficiais da ANAC.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-seed", "grao": "codigo"},
    ),
}

for tabela, (comentario, tags) in TABELAS.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    pares = ", ".join([f"'{k}' = '{v}'" for k, v in tags.items()])
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({pares})")
    print(f"{tabela}: comentario + {len(tags)} tags")



# Auditoria da Governança - 100% das colunas comentadas?

In [0]:
display(spark.sql("""
    SELECT table_name,
        COUNT(*) AS colunas,
        SUM(CASE WHEN comment IS NULL OR comment = '' THEN 1 ELSE 0 END) AS sem_comentario,
        ROUND(100.0 * SUM(CASE WHEN comment IS NOT NULL AND comment <> '' THEN 1 ELSE 0 END)
        / COUNT(*), 1) AS pct_comentado
    FROM voe_bem.information_schema.columns
    WHERE table_schema = 'silver'
    GROUP BY table_name
    ORDER BY table_name
"""))

In [0]:
display(spark.sql("SHOW TABLES IN voe_bem.silver"))